In [1]:
# Imports
import pandas as pd
import papermill as pm #papermill allows us to execute notebooks not just .py files.
import sys
import os
sys.path.append(os.path.abspath('..')) 
from data_collection import fred_data_download as fred_data_download

In [2]:
#Execute child notebooks
pm.execute_notebook('SP500_notebook.ipynb', 'SP500_notebook_output.ipynb')
pm.execute_notebook('SP500futures_notebook.ipynb', 'SP500futures_notebook_output.ipynb')
fred_data_download.run_fred_download()


Executing:   0%|          | 0/4 [00:00<?, ?cell/s]

Executing:   0%|          | 0/1 [00:00<?, ?cell/s]

            3m_treasury  2yr_treasury  10yr_treasury  yield_curve    VIX  \
date                                                                       
2025-07-30         4.41          3.94           4.38         0.44  15.48   
2025-07-31         4.41          3.94           4.37         0.43  16.72   
2025-08-01         4.35          3.69           4.23         0.54  20.38   
2025-08-02          NaN           NaN            NaN          NaN    NaN   
2025-08-03          NaN           NaN            NaN          NaN    NaN   

            fed_funds_rate  
date                        
2025-07-30            4.33  
2025-07-31            4.33  
2025-08-01            4.33  
2025-08-02            4.33  
2025-08-03            4.33  


In [3]:
# Pull DFS
sp = pd.read_parquet('../data/SP500_data.parquet')
fred = pd.read_parquet('../data/fred_data.parquet')
futures = pd.read_csv('../data/Overnight_SP500_data.csv')

In [4]:
fred.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 226 entries, 2025-07-30 to 2026-03-12
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   3m_treasury     153 non-null    float64
 1   2yr_treasury    153 non-null    float64
 2   10yr_treasury   153 non-null    float64
 3   yield_curve     154 non-null    float64
 4   VIX             159 non-null    float64
 5   fed_funds_rate  225 non-null    float64
dtypes: float64(6)
memory usage: 12.4 KB


In [5]:
sp.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 153 entries, 2025-07-31 to 2026-03-10
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Close               153 non-null    float64
 1   High                153 non-null    float64
 2   Low                 153 non-null    float64
 3   Open                153 non-null    float64
 4   Volume              153 non-null    int64  
 5   Day Change %        153 non-null    float64
 6   Overnight Change %  152 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 9.6 KB


In [6]:
fred.index.name = 'Date'

In [7]:
#Perform Left join on date to consolidate dataframes for modeling.
data = pd.merge(sp, fred, on='Date', how='left')

In [8]:
data.head()

,Close,High,Low,Open,Volume,Day Change %,Overnight Change %,3m_treasury,2yr_treasury,10yr_treasury,yield_curve,VIX,fed_funds_rate
Date,,,,,,,,,,,,,
2025-07-31,6339.390137,6427.020020,6327.640137,6427.020020,6077080000,-1.363461,NaN,4.41,3.94,4.37,0.43,16.72,4.33
2025-08-01,6238.009766,6287.279785,6212.689941,6287.279785,5827150000,-0.783646,-0.822009,4.35,3.69,4.23,0.54,20.38,4.33
2025-08-04,6329.939941,6330.689941,6271.709961,6271.709961,4842580000,0.928455,0.540240,4.35,3.69,4.22,0.53,17.52,4.33
2025-08-05,6299.189941,6346.000000,6289.370117,6336.629883,5517410000,-0.590849,0.105687,4.34,3.72,4.22,0.50,17.85,4.33
2025-08-06,6345.060059,6352.830078,6301.109863,6309.299805,5408560000,0.566786,0.160495,4.32,3.69,4.22,0.53,16.77,4.33


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 153 entries, 2025-07-31 to 2026-03-10
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Close               153 non-null    float64
 1   High                153 non-null    float64
 2   Low                 153 non-null    float64
 3   Open                153 non-null    float64
 4   Volume              153 non-null    int64  
 5   Day Change %        153 non-null    float64
 6   Overnight Change %  152 non-null    float64
 7   3m_treasury         151 non-null    float64
 8   2yr_treasury        151 non-null    float64
 9   10yr_treasury       151 non-null    float64
 10  yield_curve         151 non-null    float64
 11  VIX                 153 non-null    float64
 12  fed_funds_rate      153 non-null    float64
dtypes: float64(12), int64(1)
memory usage: 16.7 KB


In [10]:
#show null rows
data[data.isnull().any(axis=1)]
#nulls in 3m_treasury, 2 yr 10 yr, vix ,fed funds, solve with ffill
data = data.fillna(method='ffill')

/var/folders/3n/hh67zc0j3vl1gh8dpln88fv40000gn/T/ipykernel_33080/1399603370.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method='ffill')


In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 153 entries, 2025-07-31 to 2026-03-10
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Close               153 non-null    float64
 1   High                153 non-null    float64
 2   Low                 153 non-null    float64
 3   Open                153 non-null    float64
 4   Volume              153 non-null    int64  
 5   Day Change %        153 non-null    float64
 6   Overnight Change %  152 non-null    float64
 7   3m_treasury         153 non-null    float64
 8   2yr_treasury        153 non-null    float64
 9   10yr_treasury       153 non-null    float64
 10  yield_curve         153 non-null    float64
 11  VIX                 153 non-null    float64
 12  fed_funds_rate      153 non-null    float64
dtypes: float64(12), int64(1)
memory usage: 16.7 KB


In [12]:
futures.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 186 entries, 0 to 185
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Date                  186 non-null    object 
 1   Overnight_Return      186 non-null    float64
 2   Overnight_Volatility  186 non-null    float64
 3   Futures_Last_Price    186 non-null    float64
dtypes: float64(3), object(1)
memory usage: 5.9+ KB


In [13]:
#convert futures to date time index
futures['Date'] = pd.to_datetime(futures['Date'])
futures.index = futures['Date']
futures = futures.drop(columns=['Date'])

In [14]:
#Merge with consolidated data
data = pd.merge(data, futures, on='Date', how='left')

In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 153 entries, 2025-07-31 to 2026-03-10
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Close                 153 non-null    float64
 1   High                  153 non-null    float64
 2   Low                   153 non-null    float64
 3   Open                  153 non-null    float64
 4   Volume                153 non-null    int64  
 5   Day Change %          153 non-null    float64
 6   Overnight Change %    152 non-null    float64
 7   3m_treasury           153 non-null    float64
 8   2yr_treasury          153 non-null    float64
 9   10yr_treasury         153 non-null    float64
 10  yield_curve           153 non-null    float64
 11  VIX                   153 non-null    float64
 12  fed_funds_rate        153 non-null    float64
 13  Overnight_Return      151 non-null    float64
 14  Overnight_Volatility  151 non-null    float64
 15  Futu

In [16]:
fred.head()

,3m_treasury,2yr_treasury,10yr_treasury,yield_curve,VIX,fed_funds_rate
Date,,,,,,
2025-07-30,4.41,3.94,4.38,0.44,15.48,4.33
2025-07-31,4.41,3.94,4.37,0.43,16.72,4.33
2025-08-01,4.35,3.69,4.23,0.54,20.38,4.33
2025-08-02,NaN,NaN,NaN,NaN,NaN,4.33
2025-08-03,NaN,NaN,NaN,NaN,NaN,4.33


In [17]:
# A few nulls in futures, solve with ffill
data = data.fillna(method='ffill')

/var/folders/3n/hh67zc0j3vl1gh8dpln88fv40000gn/T/ipykernel_33080/899672359.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method='ffill')


In [18]:
data.reset_index(inplace=True)

In [19]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Date                  153 non-null    datetime64[ns]
 1   Close                 153 non-null    float64       
 2   High                  153 non-null    float64       
 3   Low                   153 non-null    float64       
 4   Open                  153 non-null    float64       
 5   Volume                153 non-null    int64         
 6   Day Change %          153 non-null    float64       
 7   Overnight Change %    152 non-null    float64       
 8   3m_treasury           153 non-null    float64       
 9   2yr_treasury          153 non-null    float64       
 10  10yr_treasury         153 non-null    float64       
 11  yield_curve           153 non-null    float64       
 12  VIX                   153 non-null    float64       
 13  fed_funds_rate      

In [20]:
data.to_csv('../data/consolidated_data.csv', index=True)

In [21]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Date                  153 non-null    datetime64[ns]
 1   Close                 153 non-null    float64       
 2   High                  153 non-null    float64       
 3   Low                   153 non-null    float64       
 4   Open                  153 non-null    float64       
 5   Volume                153 non-null    int64         
 6   Day Change %          153 non-null    float64       
 7   Overnight Change %    152 non-null    float64       
 8   3m_treasury           153 non-null    float64       
 9   2yr_treasury          153 non-null    float64       
 10  10yr_treasury         153 non-null    float64       
 11  yield_curve           153 non-null    float64       
 12  VIX                   153 non-null    float64       
 13  fed_funds_rate      

In [22]:
data.head()

,Date,Close,High,Low,Open,Volume,Day Change %,Overnight Change %,3m_treasury,2yr_treasury,10yr_treasury,yield_curve,VIX,fed_funds_rate,Overnight_Return,Overnight_Volatility,Futures_Last_Price
0,2025-07-31,6339.390137,6427.020020,6327.640137,6427.020020,6077080000,-1.363461,NaN,4.41,3.94,4.37,0.43,16.72,4.33,0.006948,0.002083,6456.75
1,2025-08-01,6238.009766,6287.279785,6212.689941,6287.279785,5827150000,-0.783646,-0.822009,4.35,3.69,4.23,0.54,20.38,4.33,-0.022168,0.002764,6314.75
2,2025-08-04,6329.939941,6330.689941,6271.709961,6271.709961,4842580000,0.928455,0.540240,4.35,3.69,4.22,0.53,17.52,4.33,0.006290,0.000658,6304.00
3,2025-08-05,6299.189941,6346.000000,6289.370117,6336.629883,5517410000,-0.590849,0.105687,4.34,3.72,4.22,0.50,17.85,4.33,0.009776,0.001891,6365.75
4,2025-08-06,6345.060059,6352.830078,6301.109863,6309.299805,5408560000,0.566786,0.160495,4.32,3.69,4.22,0.53,16.77,4.33,-0.003991,0.001797,6340.25
